# Extract Data from Simulations

## Imports

In [1]:
# System functions
import numpy as np
import os, sys

# Adapt accordingly
sys.path.append('/home/ws/om2854/jaxionsdir/jaxions/scripts')
os.environ["PATH"] = os.environ.get("PATH", "") + ":/usr/bin"

# Kinetic misalignment functions
from pyaxions import jaxions as pa
from pyaxions import spectrum as spec

## Load simulation data

In [2]:
# Simulation parameters
N = 1024 # Points/dimension
L = 3 # Box length in L₁ units

# Axion parameters
fAGeV = 5e9 # Axion decay constant in GeV
theta1 = 0 # Initial field value of zero mode
vheta1 = 630 # Initial velocity value of zero mode

In [3]:
# Load measurement file
path = os.getcwd()
file_path = f"/simulations/out_N{N}_L{L}_fA{fAGeV:.0e}_theta{theta1:.0f}_vheta{vheta1:.0f}"

mf = pa.findmfiles(path + file_path)
mask = pa.gml(mf, 'psp?') # Files with spectrum measurements

In [4]:
# Measurements
R = pa.gml(mf, 'R') # Scale factor in R₁
tau = pa.gml(mf, 'ct') # Conformal time 
mA = pa.gml(mf, 'massA') # Axion mass, conformal mass: ma*r
eA = pa.gml(mf, 'eA') # Axion energy
sizeN = pa.gm(mf[0], 'N') # Points/dimension
nmodes = pa.phasespacedensityBOX(sizeN) # Phase space modes (averages over # of modes in each shell)
esps = pa.gml(mf[mask], 'espV_0')/(nmodes*(R[mask, None]**2)) # Axion potential energy density spectrum
s = spec.espevol(mf) # Spectral values
k_values = s.avek # Momentum values

## Save to .txt file

### Mode Evolution

In [5]:
for idx in np.arange(len(k_values)):
    # Normalization
    norm = 1 / (mA[mask]*R[mask])**2 / (esps[0, idx]/(mA[mask][0]*R[mask][0])**2)
    # Mode evolution w.r.t. tau = 0
    esps[:, idx] = norm * esps[:, idx]

In [6]:
# Create the header row: first element empty, then tau values
tau_row = np.concatenate(([0], tau[mask])) 

# Transpose psps so that rows correspond to k_values
esps_matrix = np.column_stack([k_values, np.asarray(esps).T])  # first column = k_values, rest = psps

# Combine tau and esps
final_esps = np.vstack([tau_row, esps_matrix])

np.savetxt(f'data/mode_evol_fA{fAGeV:.0e}_theta{theta1:.0f}_vheta{vheta1:.0f}.txt', final_esps, header = '1. row: tau values; 1. column: k values; remaining: normalized mode evolution values')

### Spectrum Evolution

In [7]:
psps = []
for idx in np.arange(len(mf[mask])):
    # Normalization
    norm = k_values**3/(2*np.pi**2) / (nmodes*eA[mask][idx]**2)
    # Dimensionless axion energy density contrast spectrum
    psp = norm * pa.gm(mf[mask][idx], 'psp') 
    psps.append(psp)
    
psps = np.asarray(psps)

In [8]:
# Create the header row: first element empty, then tau values
tau_row = np.concatenate(([0], tau[mask])) 

# Transpose psps so that rows correspond to k_values
psps_matrix = np.column_stack([k_values, np.asarray(psps).T])  # first column = k_values, rest = psps

# Combine tau and psps
final_psps = np.vstack([tau_row, psps_matrix])

np.savetxt(f'data/spectrum_evol_fA{fAGeV:.0e}_theta{theta1:.0f}_vheta{vheta1:.0f}.txt', final_psps, header = '1. row: tau values; 1. column: k values; remaining: normalized power spectrum values')

## Read from .txt file

In [9]:
# Skip the first line (header)
data = np.loadtxt(f'data/spectrum_evol_fA{fAGeV:.0e}_theta{theta1:.0f}_vheta{vheta1:.0f}.txt', skiprows = 1)

# Extract tau values (first row, excluding first element)
tau = data[0, 1:]

# Extract k values (first column, excluding first row)
k = data[1:, 0]

# Extract psps matrix
psps = data[1:, 1:]

In [10]:
# Skip the first line (header)
data = np.loadtxt(f'data/mode_evol_fA{fAGeV:.0e}_theta{theta1:.0f}_vheta{vheta1:.0f}.txt', skiprows = 1)

# Extract tau values (first row, excluding first element)
tau = data[0, 1:]

# Extract k values (first column, excluding first row)
k = data[1:, 0]

# Extract esps matrix
esps = data[1:, 1:]